This is an implementation of the Algorithm $RandHeavyTest_{0.5}$ on the paper \href{https://arxiv.org/abs/2607.11203}{to be checked}. In this Travelling Salesman Problem, given a cycle graph $G = (V, E)$ with nonnegative edges and source $s$. Each vertex represents a city, each edge the possible connection between two cities, and the weight of each edge the cost to be paid when moving between two cities(This is why it is nonnegative). The objective is to visit all cities(can be visited many times) starting from the given source $s$(vertex' index) and coming back to the source with a minimum cost.

The agent does not know the graph in advance. They just discover two possibe edges(to the left or to the right) on the cycle graph at each step. They have to choose one and cannot catch-up in the future when doing the wrong choice.

The competitive ratio measure the efficiency of the algorithm. It is the cost found by the online algorithm divided by the cost of the offline algorithm noted $OPT$. The cost of the online algorithm($RandHeavyTest_{0.5}$) is given by the expectation since it is random. And $OPT = min\left(\sum_{e\in E}w(e), 2\sum_{e\in A \setminus h}w(e)\right)$ where $h$ is a heavy edge. An edge $h$ is heavy if $\sum_{e\in A \setminus h}w(e) < w(h).$

Let'say that we have the following flow at some step $t$ of the the algorithm. $s → a_t→ b_t→d_t→s.$ For this explanation, assume that $a_t$ is the path discovered by agent in the left(clockwise direction) at time $t$, $d_t$ in the counter clockwise direction (the boundary edge of $d_t$ is not visited yet) and $b_t$ the boundary edge at the branch $a_t$. It can be the opposite direction but the same principle.

At this time $t$, the agent chooses randomly between $b_t$ and the boundary edge on $d_t$ with a probability $p_{dir} = \begin{cases}
1 & \text{if } b_t \le 1.5a_t + d_t \\
\frac{a_t}{2(b_t - a_t -d_t)} & \text{otherwise }
\end{cases}$

So, at this time $t$ if the probability is not $1$, two terms with probability $p_t$ and $1 - p_t$ appear to compute the expectation. And for a given graph with $n$ vertices, the algorithm can find $O(n^2)$ possible probabilities. It is quite hard to prove the bound of the competitive ratio using this $O(n^2)$ probabilities then the paper proposed the poetential function $\Phi$ which is a supermatingale. They proved that the bound for any configuration of $G$ will give a competitive ratio less than $\frac{(3 - \sqrt{2})^2}{8}$ by analyzing different value of $b_t$.

This competitive ratio can be reduced to $1.307$ if we consider other edge in different position apart from $b_t$. They are the heaviest edge $b'$(which is the heavy edge if the cycle contains one) and $b''$ which is the second heaviest after the heaviest one. There is another proof for that one with the notion of $T_{stop}$ which will be published in another paper. For now, we just nedd the simulation.

The function $prob\_one\_step(G, N, a, b, d, ag\_pos, back\_pos, clockwise, C, rng)$ update the branches $a$, $d$, the boundary edge $b$, the cost $C$paid by the agent after one step. The cycle graph is stored in a list $Graph$. $clockwise$ informs the direction of the last move. $False$ if it was in the left and $True$ otherwise.

$Graph[i]$ refers to the edge before the vertex $i$ in the counter clockwise direction if the boolean clockwise is $False$.
It is the edge before the vertex $i$ in the clockwise direction if clockwise is $True$.

$ag\_pos$ is the index of Graph such that $Graph[ag\_pos] = b.$Similar to $b$, $back\_pos$ is the index of Graph such that $Graph[back\_pos]$ is the unvisited vertex of the branch $d$. Initially $ag\_pos = back\_pos = s.$

$prob\_one\_step$ is just computing the new cost after one step according to the random value compared with the $p\_dir$ at this step, not the expectation.

In [17]:
import numpy as np

In [18]:
def prob_one_step(G, N, a, b, d, ag_pos, back_pos, clockwise, C, rng):
  if b <= 1.5 * a + d:#Move direct
    C += b
    p = 1
    if clockwise == False:
      a = a + G[ag_pos]
      b = G[(ag_pos - 1) % N]
      ag_pos = (ag_pos - 1) % N
    else:#clockwise = True
      a = a + G[(ag_pos + 1) % N]
      b = G[(ag_pos + 2) % N]
      ag_pos = (ag_pos + 1) % N
  else:#Backtracking with probability 1 - p
    p = a / (2 * (b - a - d))
    if clockwise == False:
      #C += p * b + (1 - p) * (a + d)
      if rng.uniform(low=0, high=1) <= 1 - p:
        C += a + d
        temp = back_pos
        back_pos = ag_pos
        temp_d = d
        d = a + b
        a = temp_d#We are adding the last unvisited edge of the previous d.
        ag_pos = (temp + 1) % N
        b = G[(ag_pos + 1) % N]
        clockwise = True
      else:#go direct
        #print("Go direct")
        C += b
        a = a + G[ag_pos]
        b = G[(ag_pos - 1) % N]
        ag_pos = (ag_pos - 1) % N
    else:#clockwise is True
      if rng.uniform(low=0, high=1) <= 1 - p:
        C += a + d
        temp = back_pos
        back_pos = ag_pos
        temp_d = d
        d = a + b
        a = temp_d
        ag_pos = (temp - 1) % N
        b = G[ag_pos]
        clockwise = False
      else:#go direct
        #print("Go direct")
        C += b
        a = a + G[(ag_pos + 1) % N]
        b = G[(ag_pos + 2) % N]
        ag_pos = (ag_pos + 1) % N
  #print("p = ", p, "a = ", a, " b = ", b, " d = ", d, " ag_pos = ", ag_pos, " back_pos = ", back_pos, " clockwise = ", clockwise, "Cost = ", C)
  return G, a, b, d, ag_pos, back_pos, clockwise, C

The function $find\_opt$ compute $OPT$.

In [19]:
def find_opt(G):#G is a numpy array
  summation = np.sum(G)
  heaviest = np.max(G)
  return min(2 * (summation - heaviest), summation)

$Randheavytets0\_5(G, source)$ repeats the step $n - 2$ times where $n$ is the number of edges. $n-2$ because the algorithm choose with probability $1$ the cheapest between the $2$ proposed edges in the begining and when the last edge is discovered at the end of the algorithm, the agent goes back by choosing the cheapest branch.  But there is some required initialization to be consistent with the index and direction.

In [20]:
def Randheavytest0_5(G, source):
  N  = len(G)
  G = np.array(G)
  s = 0 #must be < N
  clockwise = False
  #Assume that when moving to the left branch, index decreases, and increases in the opposite direction.
  #First move should be to the left for reference. The array Graph is adapted to that.
  if G[source] > G[source + 1]:
    L = list(G)
    L.reverse()
    G = np.array(L)
    source = N - 1 - source
    clockwise = False

  a, b, d = G[source % N], G[(source - 1) % N], G[(source + 1) % N]#It is correct since clockwise = False
  ag_pos, back_pos = (source - 1) % N, source#definition of ag_pos and back_pos are inside the function prob_one_step
  C = a # Cost paid by the algorithm
  rng = np.random.default_rng()
  for i in range(N - 2):
    G, a, b, d, ag_pos, back_pos, clockwise, C = prob_one_step(G, N, a, b, d, ag_pos, back_pos, clockwise, C, rng)
  det = min(a, b + d)
  C += det
  OPT = find_opt(G)
  return C / OPT

For example, given the graph $Graph = [1, 2, 8, 35, 71, 18, 4]$ and the source is $0$. Running this code many times will give different comptetitive ratio since it is randomized. And it can exceed $1.314$ since the algorithm is not using the expectation. The competitive ratio found by this algorithm is the one of the real value found by the agent when exploring randomly. Then we still need another function verifying our analysis. And this function is deterministic since it computes the expectation.

In [22]:
Graph, source = [1, 2, 8, 35, 71, 18, 4], 0
Randheavytest0_5(Graph, source)

np.float64(1.4411764705882353)

The function $com\_ratio\_exp(G, source)$ is implementing the computation of a competitive ratio using the expectation. It is mentioned before that there can be $O(n^2)$ different probabilities for this task. This $O(n^2)$ is only possible if we use two lists of $n + 1$ elements called $pref\_cw$ and $pref\_ccw$.  They contain respectively the cumulative value of the cost paid by the agent from the source and making a cycle in clockwise and counterclockwise direction staring from $0$.  

In [23]:
from collections import defaultdict

In [24]:
def p_dir(a, b, d, alpha=0.5):
    if b <= (1.0 + alpha) * a + d:
        return 1.0
    return alpha * a / (b - d - a)

In [25]:
def adapting_index(G):
  L = G[1:]
  L.append(G[0])
  return L

In [26]:
def com_ratio_exp(G, source=0, alpha=0.5):
    N = len(G)
    G = adapting_index(G)
    print(source)
    if N <= 1:
        return 0.0
    if not (0 <= source < N):
        raise ValueError("start must satisfy 0 <= start < len(G)")

    w = G[source:] + G[:source]
    total = sum(w)
    pref_cw = [0.0] * (N + 1)
    pref_ccw = [0.0] * (N + 1)
    for i in range(N):
        pref_cw[i + 1] = pref_cw[i] + w[i]
        pref_ccw[i + 1] = pref_ccw[i] + w[N - 1 - i]

    if w[0] <= w[N - 1]:
        exp_cost = w[0]
        states = {(1, 0, 1): 1.0}
    else:
        exp_cost = w[N - 1]
        states = {(0, 1, 0): 1.0}
    #print(states)

    for _ in range(N - 2):
        #print("_ = ", _)
        next_states = defaultdict(float)
        for (r, l, side), prob in states.items():
            #print("side = ", side)
            if side == 1:
                a = pref_cw[r]
                b = w[r]
                d = pref_ccw[l] + w[N - 1 - l]
                direct_state = (r + 1, l, 1)
                back_state = (r, l + 1, 0)
                #print("direct_state = ", direct_state)
                #print("back_state = ", back_state)
            else:
                a = pref_ccw[l]
                b = w[N - 1 - l]
                d = pref_cw[r] + w[r]
                direct_state = (r, l + 1, 0)
                back_state = (r + 1, l, 1)
                #print("direct_state = ", direct_state)
                #print("back_state = ", back_state)

            p = p_dir(a, b, d, alpha)
            exp_cost += prob * (p * b + (1.0 - p) * (a + d))
            #print(p)
            if p > 0.0:
                next_states[direct_state] += prob * p
                #print(next_states[direct_state])
            if p < 1.0:
                next_states[back_state] += prob * (1.0 - p)
                #print(next_states[back_state])
        states = next_states
        #print("states = ", states)

    for (r, l, side), prob in states.items():
        dist = pref_cw[r] if side == 1 else pref_ccw[l]
        exp_cost += prob * min(dist, total - dist)

    OPT = find_opt(np.array(w))
    ratio = exp_cost / OPT
    return ratio

In [27]:
com_ratio_exp([1, 2, 8, 35, 71, 18, 4], 0)

0


np.float64(1.1110879010695187)